In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## データ前処理

In [22]:
# 学習データの読み込み
df_train = pd.read_csv('train.csv')
# 目的変数であるSalePriceを抽出
df_train_y = df_train['SalePrice']
# 目的変数を削除
df_train = df_train.drop('SalePrice', axis=1)
# idを削除
df_train = df_train.drop('Id', axis=1)

In [23]:
missing_columns = df_train.columns[df_train.isnull().sum() > 0]
print(missing_columns)

Index(['LotFrontage', 'Alley', 'MasVnrType', 'MasVnrArea', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Electrical', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence',
       'MiscFeature'],
      dtype='object')


In [24]:
df_train_no_missing = df_train.drop(missing_columns, axis=1)

In [25]:
df_train_no_missing.columns

Index(['MSSubClass', 'MSZoning', 'LotArea', 'Street', 'LotShape',
       'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood',
       'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl',
       'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'GarageCars', 'GarageArea', 'PavedDrive',
       'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
       'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SaleType',
       'SaleCondition'],
      dtype='object')

In [26]:
# カテゴリカル変数の抽出
categorical_columns = df_train_no_missing.columns[df_train_no_missing.dtypes == 'object']
print(categorical_columns)

Index(['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating',
       'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive',
       'SaleType', 'SaleCondition'],
      dtype='object')


In [27]:
# カテゴリカル変数の列を削除
df_train_no_missing_no_categorical = df_train_no_missing.drop(categorical_columns, axis=1)
df_train_no_missing_no_categorical.columns

Index(['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt',
       'YearRemodAdd', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
       '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath',
       'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr',
       'TotRmsAbvGrd', 'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF',
       'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal', 'MoSold', 'YrSold'],
      dtype='object')

#### フィルター法

In [39]:
# フィルター法による特徴量選択
from sklearn.feature_selection import SelectKBest, f_regression, VarianceThreshold

# 分散が小さい特徴量を削除
threshold_value = 100
selector = VarianceThreshold(threshold=threshold_value)
selector.fit_transform(df_train_no_missing_no_categorical)

array([[   60,  8450,  2003, ...,     0,     0,     0],
       [   20,  9600,  1976, ...,     0,     0,     0],
       [   60, 11250,  2001, ...,     0,     0,     0],
       ...,
       [   70,  9042,  1941, ...,     0,     0,  2500],
       [   20,  9717,  1950, ...,     0,     0,     0],
       [   20,  9937,  1965, ...,     0,     0,     0]])

In [40]:
df_train_new = pd.DataFrame(selector.fit_transform(df_train_no_missing_no_categorical), columns=df_train_no_missing_no_categorical.columns[selector.get_support()])
df_train_new.columns

Index(['MSSubClass', 'LotArea', 'YearBuilt', 'YearRemodAdd', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF',
       'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal'],
      dtype='object')

In [45]:
### 変動係数を用いた特徴量選択 ###

# 変動係数を計算
cv = df_train_no_missing_no_categorical.std() / df_train_no_missing_no_categorical.mean()

# 閾値を設定
threshold_value = 1

# 閾値以上の特徴量を抽出
selected_features = df_train_no_missing_no_categorical.columns[cv > threshold_value]
df_selected = df_train_no_missing_no_categorical[selected_features]

print(f"選択された特徴量: {selected_features}")

選択された特徴量: Index(['BsmtFinSF1', 'BsmtFinSF2', '2ndFlrSF', 'LowQualFinSF', 'BsmtFullBath',
       'BsmtHalfBath', 'HalfBath', 'Fireplaces', 'WoodDeckSF', 'OpenPorchSF',
       'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal'],
      dtype='object')


#### 多重共線性の軽減

In [74]:
df_train_no_missing_no_categorical[selected_features].head()

,BsmtFinSF1,BsmtFinSF2,2ndFlrSF,LowQualFinSF,BsmtFullBath,BsmtHalfBath,HalfBath,Fireplaces,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal
0,706,0,854,0,1,0,1,0,0,61,0,0,0,0,0
1,978,0,0,0,0,1,0,1,298,0,0,0,0,0,0
2,486,0,866,0,1,0,1,1,0,42,0,0,0,0,0
3,216,0,756,0,1,0,0,1,0,35,272,0,0,0,0
4,655,0,1053,0,1,0,1,1,192,84,0,0,0,0,0


In [ ]:
# 変数間の相関係数を計算
from seaborn import heatmap
correlation = df_train_no_missing_no_categorical.corr()

## モデルの作成

### 重回帰モデル

In [68]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

# 欠損値の補完
imputer = SimpleImputer(strategy='mean')
# df_selected = pd.DataFrame(imputer.fit_transform(df_selected), columns=df_selected.columns) # カラム名を指定しないとカラム名が消えてしまう
df_selected = pd.DataFrame(imputer.fit_transform(df_train_no_missing_no_categorical), columns=df_train_no_missing_no_categorical.columns)

# データの分割
X_train, X_test, y_train, y_test = train_test_split(df_selected, df_train_y, test_size=0.5, random_state=0)

# train_test_splitではstraitfyという引数を指定することで、層化抽出を行うことができる
# この引数を指定することで、元のデータのクラスの比率を保ったまま、データを分割することができる

# モデルの作成
model = LinearRegression()
model.fit(X_train, y_train)

# 予測
y_pred = model.predict(X_test)

# 評価
mse = mean_squared_error(y_test, y_pred)
print(f"MSE: {mse}")

MSE: 1840689529.7711031


In [71]:
# テストデータの読み込み
df_test = pd.read_csv('test.csv')
# df_test = df_test[selected_features]
df_test = df_test[df_train_no_missing_no_categorical.columns]
df_test = pd.DataFrame(imputer.transform(df_test), columns=df_test.columns)

In [72]:
y_pred = model.predict(df_test)

In [73]:
# 提出用ファイルの作成
submission = pd.read_csv('sample_submission.csv')
submission['SalePrice'] = y_pred
submission.to_csv('test_submission.csv', index=False)